In [ ]:
# Import libraries
import pandas as pd

In [4]:
# Read the Notebook 1 
df=pd.read_csv(r"D:\Olist\olist_ml_dataset.csv")
df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,avg_item_price,max_item_price,min_item_price,total_freight,total_items,seller_id,total_pyment_value,max_installments,primary_pyment_type,avg_review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,29.99,29.99,29.99,8.72,1.0,3504c0cb71d7fa48d967e0e4c94d59d9,38.71,1.0,voucher,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,118.70,118.70,118.70,22.76,1.0,289cdb325fb7e7f891c38608bf9e0962,141.46,1.0,boleto,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,159.90,159.90,159.90,19.22,1.0,4869f7a5dfa277a7dca6462dcf3b52b2,179.12,3.0,credit_card,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,59296,...,45.00,45.00,45.00,27.20,1.0,66922902710d126a0e7d26b0e3805106,72.20,1.0,credit_card,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,19.90,19.90,19.90,8.72,1.0,2c9e548be18521d1c43cde1c582c6de8,28.62,1.0,credit_card,5.0


### Build the label from the delivery date against the estimated delivery date

In [8]:
# Parse data columns to datatime objects
df["order_delivered_customer_date"]=pd.to_datetime(df["order_delivered_customer_date"])
df["order_estimated_delivery_date"]=pd.to_datetime(df["order_estimated_delivery_date"])

# Build the label (1 if late, 0 if on time)
df['is_late']=(df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]).astype(int)


### Check the label is correct on a few real orders before you trust it

In [11]:
sample_check=df[["order_delivered_customer_date", "order_estimated_delivery_date", "is_late"]].tail(10)
sample_check

,order_delivered_customer_date,order_estimated_delivery_date,is_late
99431,2017-11-10 17:57:22,2017-11-22,0
99432,2018-01-26 15:45:14,2018-01-18,1
99433,2017-10-20 20:25:45,2017-11-07,0
99434,2017-02-07 13:15:25,2017-03-17,0
99435,2017-03-06 11:08:08,2017-03-22,0
99436,2017-03-17 15:08:01,2017-03-28,0
99437,2018-02-28 17:37:56,2018-03-02,0
99438,2017-09-21 11:24:17,2017-09-27,0
99439,2018-01-25 23:32:54,2018-02-15,0
99440,2018-03-16 13:08:30,2018-04-03,0


### Look at the class distribution — how many late, how many on time

In [ ]:
df['is_late'].value_counts(normalize=True)

is_late
0    0.92129
1    0.07871
Name: proportion, dtype: float64

### Decide if you have a class imbalance problem, and how big it is

In [30]:
late_count=(df['is_late']==1).sum()
on_time_count=(df['is_late']==0).sum()

print(f"On Time Count= {on_time_count}")
print(f"Late Count   = {late_count}")


On Time Count= 91614
Late Count   = 7827


### Decide if you have a class imbalance problem, and how big it is

### Yes, there is a severe class imbalance problem.

Problem Size in Numbers:

  1 Majority Class (On Time - 0): Represents 92.13% of the data (91,614 orders).

  2  Minority Class (Late - 1): Represents only 7.87% of the data (7,827 orders).

The ratio between the two classes is approximately 12 : 1 (for every 1 late order, there are 12 on-time orders).

### Artifact: the labeled table

In [31]:
df.to_csv("ml_dataset_labeled.csv",index=False)


In [32]:
print(f"\nArtifact Shape: {df.shape}")
df.head()


Artifact Shape: (99441, 24)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,max_item_price,min_item_price,total_freight,total_items,seller_id,total_pyment_value,max_installments,primary_pyment_type,avg_review_score,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,29.99,29.99,8.72,1.0,3504c0cb71d7fa48d967e0e4c94d59d9,38.71,1.0,voucher,4.0,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,118.70,118.70,22.76,1.0,289cdb325fb7e7f891c38608bf9e0962,141.46,1.0,boleto,4.0,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,159.90,159.90,19.22,1.0,4869f7a5dfa277a7dca6462dcf3b52b2,179.12,3.0,credit_card,5.0,0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,45.00,45.00,27.20,1.0,66922902710d126a0e7d26b0e3805106,72.20,1.0,credit_card,5.0,0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,19.90,19.90,8.72,1.0,2c9e548be18521d1c43cde1c582c6de8,28.62,1.0,credit_card,5.0,0
